# PDF Ingestion

This notebook extracts structured text from the sample PDFs and stores the result in an `AcademicDB` under a named strategy.


## Setup

PDF parsing can take a minute on the first run because optional parser dependencies may initialize caches.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
import tempfile
from textwrap import shorten


def find_project_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "episcope").exists():
            return candidate
    return None


PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is not None:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

pdf_candidates = [
    Path.cwd() / "pdf_samples",
    Path.cwd() / "notebooks" / "pdf_samples",
]
if PROJECT_ROOT is not None:
    pdf_candidates.append(PROJECT_ROOT / "notebooks" / "pdf_samples")

PDF_DIR = next((path for path in pdf_candidates if path.exists()), None)
if PDF_DIR is None:
    raise FileNotFoundError("Could not find the pdf_samples directory.")

pdf_paths = sorted(PDF_DIR.glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError(f"No PDF files found in {PDF_DIR}.")

WORK_DIR = Path(tempfile.mkdtemp(prefix="episcope-pdf-"))
[path.name for path in pdf_paths], WORK_DIR


## Create A Loader And Store

`DocumentLoaderFactory` selects the parser. `InMemoryAcademicDB` is a local store that can persist to JSON for repeatable experiments.


In [ ]:
from episcope.db.in_memory_academic_db import InMemoryAcademicDB
from episcope.rag.ingestion.document_loader import DocumentLoaderFactory

strategy_name = "pdf-samples-unstructured"
loader = DocumentLoaderFactory.get_loader("unstructured")
db = InMemoryAcademicDB(backup_file=str(WORK_DIR / "academic_db.json"))

loader.__class__.__name__, strategy_name


## Extract And Persist Records

Each paper is stored as three record types: `sections`, `metadata`, and `references`. Some loaders or PDFs may return no references; the sections and metadata are still enough for indexing and retrieval.


In [ ]:
def persist_pdf(pdf_path: Path) -> dict[str, object]:
    sections, metadata, references = loader.load(pdf_path)
    metadata.file_path = str(pdf_path)

    doc_id = pdf_path.stem
    metadata_record = metadata.to_dict()
    metadata_record["original_paper_id"] = doc_id

    db.insert(doc_id, "sections", strategy_name, [section.to_dict() for section in sections])
    db.insert(doc_id, "metadata", strategy_name, metadata_record)
    db.insert(doc_id, "references", strategy_name, [ref.to_dict() for ref in references])

    return {
        "doc_id": doc_id,
        "sections": len(sections),
        "references": len(references),
        "characters": sum(len(section.content) for section in sections),
        "title": metadata.title,
    }


summary = [persist_pdf(path) for path in pdf_paths]
summary


## Inspect Stored Content

Use the same `strategy_name` whenever you retrieve records created by this ingestion run.


In [ ]:
doc_ids = db.list_docs(strategy_name)

for doc_id in doc_ids:
    sections = db.retrieve(doc_id, "sections", strategy_name) or []
    metadata = db.retrieve(doc_id, "metadata", strategy_name)
    references = db.retrieve(doc_id, "references", strategy_name) or []
    total_chars = sum(len(section.content) for section in sections)
    print(f"{doc_id}")
    print(f"  title: {metadata.title}")
    print(f"  sections: {len(sections)}")
    print(f"  references: {len(references)}")
    print(f"  text characters: {total_chars:,}")


In [ ]:
first_doc = doc_ids[0]
first_sections = db.retrieve(first_doc, "sections", strategy_name) or []

[
    {
        "section_title": section.title or "(untitled)",
        "section_type": section.section_type,
        "preview": shorten(section.content.replace("\n", " "), width=260),
    }
    for section in first_sections[:5]
]


## Reload A Persisted Store

The backup file lets you restart a notebook and continue from the extracted records.


In [ ]:
reloaded_db = InMemoryAcademicDB(backup_file=str(WORK_DIR / "academic_db.json"))
reloaded_db.list_docs(strategy_name)


## Checkpoint

A successful ingestion run gives you stable document ids, structured sections, metadata, and optional references. The next step is to index the sections for retrieval.
